In [ ]:
import requests
import pandas as pd
import json
import os
import re

# Configurações da API Local
BASE_URL = "http://localhost:3003"
API_KEY = "UgdRwHpekKRWafd+gA0Q1I72iCeQuKAqArrMEkU43f6UdgOJDsUnqoMKDB+2hhk8st9DNW9gSozSUdvKGI2w89RMMlMGEAEb1hYILnjroouAAMvAvZygGORoaX3DYQi4Dj8KX0mtHRcYRS7XW76BOxpijFVUa3IdhYLhZJIL1uo="
HEADERS = {
    "x-api-key": API_KEY,
    "Content-Type": "application/json"
}
ENDPOINTS = {
    "registration": f"{BASE_URL}/registration",
    "participation": f"{BASE_URL}/business-participation"
}
SCRIPT_NAME = "get-partners"

print("Ambiente configurado com sucesso.")

In [ ]:
input_path = os.path.join("..", "input", "get_partners_documentos.json")

with open(input_path, "r", encoding="utf-8") as f:
    document_list = json.load(f)

print(f"Total de documentos a processar: {len(document_list)}")

In [ ]:
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Cores pastéis por grupo de coluna: (header_hex, data_hex)
# Amarelo  → identidade (Nome, Documento)
# Azul     → dados da empresa (Razão Social, Nome Fantasia)
# Laranja  → contato e atividade (Telefones, Emails, CNAE)
# Verde    → situação fiscal (Status da Empresa)
# Cinza    → métrica (Qtd Sócios Ativos)
COLORS_EMPRESAS = {
    "Nome":              ("FFD966", "FFFCE8"),
    "Documento":         ("FFD966", "FFFCE8"),
    "Razão Social":      ("9DC3E6", "EBF3FB"),
    "Nome Fantasia":     ("9DC3E6", "EBF3FB"),
    "Telefones":         ("F4B183", "FEF3EA"),
    "Emails":            ("F4B183", "FEF3EA"),
    "CNAE":              ("F4B183", "FEF3EA"),
    "Status da Empresa": ("A9D18E", "EEF5E9"),
    "Qtd Sócios Ativos": ("C0C0C0", "F5F5F5"),
}
COLORS_SOCIOS = {
    "Nome":               ("FFD966", "FFFCE8"),
    "Documento Empresa":  ("9DC3E6", "EBF3FB"),
    "Documento do Sócio": ("C9A0DC", "F5EEF8"),
    "Nome do Sócio":      ("C9A0DC", "F5EEF8"),
}
COLORS_ERROS = {
    "Nome":      ("FFD966", "FFFCE8"),
    "Documento": ("FFB3B3", "FFF0F0"),
}

THIN_BORDER = Border(
    left=Side(style="thin", color="D0D0D0"),
    right=Side(style="thin", color="D0D0D0"),
    top=Side(style="thin", color="D0D0D0"),
    bottom=Side(style="thin", color="D0D0D0"),
)

CACHE_DIR = os.path.join("..", "responses", SCRIPT_NAME, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)


def normalize_doc(document):
    return re.sub(r"\D", "", document)


def format_sheet(ws, df, col_colors):
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    data_align = Alignment(horizontal="left", vertical="center", wrap_text=False)

    for col_idx, col_name in enumerate(df.columns, start=1):
        header_hex, data_hex = col_colors.get(col_name, ("D9D9D9", "F5F5F5"))
        header_fill = PatternFill("solid", fgColor=header_hex)
        data_fill = PatternFill("solid", fgColor=data_hex)

        header_cell = ws.cell(row=1, column=col_idx)
        header_cell.fill = header_fill
        header_cell.font = Font(bold=True, color="3B3B3B", size=10)
        header_cell.alignment = header_align
        header_cell.border = THIN_BORDER

        for row_idx in range(2, ws.max_row + 1):
            cell = ws.cell(row=row_idx, column=col_idx)
            cell.fill = data_fill
            cell.border = THIN_BORDER
            cell.font = Font(size=10)
            cell.alignment = data_align

    ws.row_dimensions[1].height = 32
    for col_idx, col_name in enumerate(df.columns, start=1):
        col_letter = get_column_letter(col_idx)
        max_content = max(
            len(str(col_name)),
            max(
                (len(str(ws.cell(row=r, column=col_idx).value or "")) for r in range(2, ws.max_row + 1)),
                default=0,
            ),
        )
        ws.column_dimensions[col_letter].width = min(max_content + 3, 55)

    ws.freeze_panes = "A2"


def fetch_company_data(nome, document):
    doc_key = normalize_doc(document)
    cache_file = os.path.join(CACHE_DIR, f"{doc_key}.json")

    if os.path.exists(cache_file):
        print(f"  -> Cache encontrado, pulando chamada à API")
        with open(cache_file, "r", encoding="utf-8") as f:
            cached = json.load(f)
        return cached["registration"], cached["participation"]

    body = {
        "identifier": 1,
        "document": doc_key,
        "webhook_url": "https://webhook.site/c1a70033-31b9-4879-a4e2-6f7ffbb7bd40",
        "async": False
    }
    try:
        print("  -> Consultando /registration...")
        reg_response = requests.post(ENDPOINTS["registration"], headers=HEADERS, json=body)
        reg_response.raise_for_status()
        reg_payload = reg_response.json()

        print("  -> Consultando /business-participation...")
        part_response = requests.post(ENDPOINTS["participation"], headers=HEADERS, json=body)
        part_response.raise_for_status()
        part_payload = part_response.json()

        with open(cache_file, "w", encoding="utf-8") as f:
            json.dump({"registration": reg_payload, "participation": part_payload}, f, ensure_ascii=False, indent=2)

        return reg_payload, part_payload

    except requests.exceptions.RequestException as e:
        print(f"  -> Erro na chamada da API: {e}")
        return None, None


def process_data(nome, reg_payload, part_payload):
    reg_data = reg_payload.get("data", [{}])[0]
    part_data = part_payload.get("data", [{}])[0].get("response", {})
    doc_inicial = reg_data.get("document", "N/A")

    telefones_raw = reg_data.get("phones", [])
    telefones = [str(t.get("phone")) for t in telefones_raw if t.get("phone")]

    emails_raw = reg_data.get("emails", [])
    emails = [str(e.get("email")) for e in emails_raw if e.get("email")]

    socios_raw = part_data.get("partners", [])
    lista_socios_detalhada = []

    for socio in socios_raw:
        status_socio = str(socio.get("status", "")).strip().lower()
        if status_socio not in ["inativo", "inativa"]:
            lista_socios_detalhada.append({
                "Nome":               nome,
                "Documento Empresa":  doc_inicial,
                "Documento do Sócio": socio.get("document", "N/A"),
                "Nome do Sócio":      socio.get("name", "Desconhecido"),
            })

    resumo = {
        "Nome":              nome,
        "Documento":         doc_inicial,
        "Razão Social":      reg_data.get("name", "N/A"),
        "Nome Fantasia":     reg_data.get("fantasy_name", "N/A"),
        "Telefones":         ", ".join(telefones) if telefones else "N/A",
        "Emails":            ", ".join(emails) if emails else "N/A",
        "CNAE":              reg_data.get("economic_activity", "N/A"),
        "Status da Empresa": reg_data.get("fiscal_situation", "N/A"),
        "Qtd Sócios Ativos": len(lista_socios_detalhada),
    }

    df_resumo = pd.DataFrame([resumo])
    df_socios = pd.DataFrame(lista_socios_detalhada)

    return df_resumo, df_socios


print("Funções e esquema de cores definidos. Cache em:", CACHE_DIR)

In [ ]:
todos_resumos = []
todos_socios = []
documentos_com_erro = []

for entry in document_list:
    nome = entry.get("nome", "")
    documento = entry.get("documento", "")
    print(f"Processando: {nome} ({documento})")
    reg, part = fetch_company_data(nome, documento)

    if reg and part:
        df_res, df_soc = process_data(nome, reg, part)
        todos_resumos.append(df_res)
        todos_socios.append(df_soc)
    else:
        print(f"  -> Falha ao extrair dados de: {documento}")
        documentos_com_erro.append({"Nome": nome, "Documento": documento})

print("\n--- Gerando Arquivo Final ---")

df_final_empresas = pd.concat(todos_resumos, ignore_index=True) if todos_resumos else pd.DataFrame(
    columns=["Nome", "Documento", "Razão Social", "Nome Fantasia", "Telefones", "Emails", "CNAE", "Status da Empresa", "Qtd Sócios Ativos"]
)
df_final_socios = pd.concat(todos_socios, ignore_index=True) if todos_socios else pd.DataFrame(
    columns=["Nome", "Documento Empresa", "Documento do Sócio", "Nome do Sócio"]
)
df_final_erros = pd.DataFrame(documentos_com_erro) if documentos_com_erro else pd.DataFrame(
    columns=["Nome", "Documento"]
)

output_dir = os.path.join("..", "responses", SCRIPT_NAME)
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "relatorio_consolidado.xlsx")

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_final_empresas.to_excel(writer, sheet_name="Empresas", index=False)
    df_final_socios.to_excel(writer, sheet_name="Sócios Ativos", index=False)
    df_final_erros.to_excel(writer, sheet_name="Erros ou Inválidos", index=False)
    format_sheet(writer.sheets["Empresas"], df_final_empresas, COLORS_EMPRESAS)
    format_sheet(writer.sheets["Sócios Ativos"], df_final_socios, COLORS_SOCIOS)
    format_sheet(writer.sheets["Erros ou Inválidos"], df_final_erros, COLORS_ERROS)

total_buscados = len(document_list)
total_sucesso = len(todos_resumos)
total_erros = len(documentos_com_erro)

print(f"\n{'='*50}")
print(f"RESUMO DA EXECUÇÃO")
print(f"{'='*50}")
print(f"Total de documentos buscados : {total_buscados}")
print(f"Processados com sucesso      : {total_sucesso}")
print(f"Erros na API                 : {total_erros}")
if documentos_com_erro:
    print("\nDocumentos com erro:")
    for e in documentos_com_erro:
        print(f"  - {e['Nome']} ({e['Documento']})")
print(f"{'='*50}")
print(f"\nArquivo salvo em: {output_file}")